# JEPA Notebook Demo: V-JEPA Future Selection, Real-World Video, and Grounded Agent Commentary

This notebook keeps the existing Moving MNIST control benchmark intact, then extends the repo into a more realistic real-video future-selection demo. The system still uses frozen V-JEPA embeddings plus the engineered compatibility scorer as the primary decision mechanism, but now it can cache a lightweight real-world dataset subset, score candidate futures, generate grounded commentary, and run a small live agent loop one example at a time.

## 1. Environment and Runtime Checks

These cells attach the repository inside the Colab runtime, install dependencies conservatively, and confirm the GPU/runtime state before any model work begins.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

DEFAULT_REPO_URL = 'https://github.com/Praveen-Rangavajhula/jepa-representation-learning.git'
REPO_ROOT_HINT = os.environ.get('JEPA_REPO_ROOT', '').strip()
REPO_URL = os.environ.get('JEPA_REPO_URL', DEFAULT_REPO_URL).strip()
REPO_BRANCH = os.environ.get('JEPA_REPO_BRANCH', '').strip()
AUTO_UPDATE_REPO = os.environ.get('JEPA_AUTO_UPDATE_REPO', '1').strip().lower() in {'1', 'true', 'yes'}
MOUNT_DRIVE = False

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

def looks_like_repo_root(path: Path) -> bool:
    return (path / 'src' / 'jepa').exists() and (path / 'notebooks').exists()

def discover_existing_repo() -> Path | None:
    candidates = []
    if REPO_ROOT_HINT:
        candidates.append(Path(REPO_ROOT_HINT).expanduser())

    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])

    for root in (Path('/content'), Path('/content/drive/MyDrive')):
        if not root.exists():
            continue
        candidates.append(root)
        try:
            for child in sorted(root.iterdir()):
                if child.is_dir():
                    candidates.append(child)
        except Exception:
            pass

    seen = set()
    for candidate in candidates:
        try:
            candidate = candidate.resolve()
        except Exception:
            continue
        key = str(candidate)
        if key in seen:
            continue
        seen.add(key)
        if looks_like_repo_root(candidate):
            return candidate
    return None

def update_repo_if_requested(repo_root: Path) -> Path:
    if not AUTO_UPDATE_REPO:
        return repo_root.resolve()
    if not str(repo_root).startswith('/content'):
        return repo_root.resolve()
    if not (repo_root / '.git').exists():
        return repo_root.resolve()

    if REPO_BRANCH:
        subprocess.check_call(['git', '-C', str(repo_root), 'checkout', REPO_BRANCH])
    print(f'Updating existing cloned repo at {repo_root}')
    subprocess.check_call(['git', '-C', str(repo_root), 'pull', '--ff-only'])
    return repo_root.resolve()

def clone_repo_if_needed() -> Path | None:
    if not REPO_URL:
        return None
    repo_name = Path(REPO_URL.rstrip('/')).stem
    if repo_name.endswith('.git'):
        repo_name = repo_name[:-4]
    target_root = Path('/content') / repo_name

    if looks_like_repo_root(target_root):
        print(f'Reusing existing cloned repo: {target_root}')
        return update_repo_if_requested(target_root)

    command = ['git', 'clone']
    if REPO_BRANCH:
        command.extend(['--branch', REPO_BRANCH])
    command.extend([REPO_URL, str(target_root)])
    print('Cloning repo:', ' '.join(command))
    subprocess.check_call(command)
    return target_root.resolve()

REPO_ROOT = discover_existing_repo()
if REPO_ROOT is not None:
    REPO_ROOT = update_repo_if_requested(REPO_ROOT)
if REPO_ROOT is None:
    REPO_ROOT = clone_repo_if_needed()

if REPO_ROOT is None:
    raise FileNotFoundError(
        'Could not find the repository in the Colab runtime. Set JEPA_REPO_ROOT or let the notebook clone it.'
    )

SRC_ROOT = REPO_ROOT / 'src'
assert (SRC_ROOT / 'jepa').exists(), f'Expected package directory at {SRC_ROOT / "jepa"}'

print(f'Repository root: {REPO_ROOT}')
print(f'Source root: {SRC_ROOT}')
print(f'Auto-update existing clone: {AUTO_UPDATE_REPO}')
%cd $REPO_ROOT
print(f'Python executable: {sys.executable}')
print(f'Working directory: {Path.cwd()}')

In [ ]:
import importlib.util
import subprocess
import sys

requirements_file = REPO_ROOT / 'requirements-colab.txt'
core_specs = {
    'torch': 'torch>=2.6,<2.8',
    'torchvision': 'torchvision>=0.21,<0.23',
}
USE_TORCH_HUB_FALLBACK = os.environ.get('JEPA_USE_TORCH_HUB_FALLBACK', '0').strip().lower() in {'1', 'true', 'yes'}
missing_core = [name for name in core_specs if importlib.util.find_spec(name) is None]

if missing_core:
    print('Installing missing core packages:', missing_core)
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', *[core_specs[name] for name in missing_core]],
        cwd=str(REPO_ROOT),
    )
else:
    print('Core packages already available: torch, torchvision')

print(f'Installing notebook dependencies from {requirements_file.name} (safe to rerun).')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(requirements_file)], cwd=str(REPO_ROOT))

if USE_TORCH_HUB_FALLBACK:
    print('Installing fallback torch.hub extras: timm, einops')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'timm', 'einops'], cwd=str(REPO_ROOT))

print('If this cell upgraded already-imported packages, restart the runtime once before continuing.')

In [ ]:
from importlib import metadata as importlib_metadata
import json
import torch

HF_CACHE_DIR = Path(os.environ.get('HF_HOME', '/content/.cache/huggingface'))
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)

versions = {}
for name in ['torch', 'torchvision', 'transformers', 'accelerate', 'huggingface_hub', 'safetensors']:
    try:
        versions[name] = importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        versions[name] = None

print('Runtime versions:')
print(json.dumps(versions, indent=2))
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA device count: {torch.cuda.device_count()}')
    print(f'Current device: {torch.cuda.get_device_name(0)}')
print(f'Hugging Face cache dir: {HF_CACHE_DIR}')
print(f'Fallback torch.hub enabled: {USE_TORCH_HUB_FALLBACK}')

## 2. Moving MNIST Control Benchmark

Moving MNIST remains the cheap, controlled benchmark. These cells still build 16-frame sequences, split them into observed prefixes and candidate futures, and save a few visual artifacts so we can compare the real-video path against a fully known synthetic setup.

In [ ]:
import importlib
import sys

if str(SRC_ROOT.resolve()) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT.resolve()))
importlib.invalidate_caches()

from jepa.commentary import DeterministicCommentaryGenerator, LLMReadyCommentaryBuilder
from jepa.data import (
    MovingMNISTConfig,
    RealVideoDataError,
    RealVideoDependencyError,
    RealVideoSubsetConfig,
    SomethingSomethingV2SubsetAdapter,
    UCF101SubsetAdapter,
    available_real_video_templates,
    create_moving_mnist_dataloaders,
    make_ucf101_fallback_config,
    save_sample_visualizations,
    summarize_batch,
)
from jepa.llm import build_default_commentary_service, generate_commentary
from jepa.agents import run_future_selection_pipeline
from jepa.models import VJEPA2Adapter, VJEPA2AdapterConfig
from jepa.scoring import VJEPAFutureScorer
from jepa.tasks import (
    FutureSelectionConfig,
    FutureSelectionDataset,
    save_future_selection_examples,
    summarize_future_selection_example,
)
from jepa.tools import (
    build_baseline_comparison_table,
    build_candidate_score_table,
    build_future_selection_context,
    build_ranking_summary,
    run_future_selection_benchmark,
    run_live_agent_loop,
    save_future_selection_benchmark_artifacts,
)

moving_config = MovingMNISTConfig(
    sequence_length=16,
    image_size=64,
    num_digits=2,
    velocity_range=(1.5, 3.5),
    train_size=512,
    test_size=128,
    mnist_root=str(REPO_ROOT / 'data'),
    seed=42,
)

task_config = FutureSelectionConfig(
    sequence_length=16,
    observed_length=8,
    future_length=8,
    num_candidates=4,
    seed=7,
)

dataset_artifacts = create_moving_mnist_dataloaders(
    moving_config,
    batch_size=8,
    num_workers=0,
    pin_memory=False,
)
train_dataset = dataset_artifacts['train_dataset']
test_dataset = dataset_artifacts['test_dataset']
train_loader = dataset_artifacts['train_loader']

train_task_dataset = FutureSelectionDataset(train_dataset, config=task_config)
test_task_dataset = FutureSelectionDataset(test_dataset, config=task_config)
example = train_task_dataset[0]
moving_batch = next(iter(train_loader))

real_video_config = RealVideoSubsetConfig(
    cache_root=os.environ.get('JEPA_REAL_VIDEO_CACHE_ROOT', 'data/real_video_cache'),
    image_size=int(os.environ.get('JEPA_REAL_VIDEO_IMAGE_SIZE', '128')),
    train_examples_per_template=int(os.environ.get('JEPA_REAL_VIDEO_TRAIN_PER_TEMPLATE', '24')),
    eval_examples_per_template=int(os.environ.get('JEPA_REAL_VIDEO_EVAL_PER_TEMPLATE', '16')),
    confident_eval_examples_per_template=int(os.environ.get('JEPA_REAL_VIDEO_CONFIDENT_PER_TEMPLATE', '40')),
    max_source_scan=int(os.environ.get('JEPA_REAL_VIDEO_MAX_SCAN', '20000')),
    local_manifest_path=os.environ.get('JEPA_REAL_VIDEO_MANIFEST') or None,
    streaming=os.environ.get('JEPA_REAL_VIDEO_STREAMING', '0').strip().lower() in {'1', 'true', 'yes'},
)

moving_results_dir = REPO_ROOT / 'results' / 'moving_mnist'
task_results_dir = REPO_ROOT / 'results' / 'task_examples'
vjepa_results_dir = REPO_ROOT / 'results' / 'vjepa_eval'
real_video_results_dir = REPO_ROOT / 'results' / 'real_video_eval'
agent_live_results_dir = REPO_ROOT / 'results' / 'agent_live'
artifact_export_dir = REPO_ROOT / 'results' / 'artifact_exports'
report_dir = REPO_ROOT / 'report'
for path in (
    moving_results_dir,
    task_results_dir,
    vjepa_results_dir,
    real_video_results_dir,
    agent_live_results_dir,
    artifact_export_dir,
    report_dir,
):
    path.mkdir(parents=True, exist_ok=True)

moving_paths = save_sample_visualizations(train_dataset, moving_results_dir, sample_indices=(0, 1), max_frames=8)
task_paths = save_future_selection_examples([train_task_dataset[i] for i in range(2)], task_results_dir, max_examples=2)

commentary_generator = DeterministicCommentaryGenerator()
llm_ready_builder = LLMReadyCommentaryBuilder()
commentary_service = build_default_commentary_service()

print('Moving MNIST batch summary:')
print(summarize_batch(moving_batch))
print('\nTask example summary:')
print(summarize_future_selection_example(example))
print('\nConfigured real-video templates:')
for template in available_real_video_templates(real_video_config):
    print(f'- {template}')
print('\nSaved data artifacts:')
for path in [*moving_paths, *task_paths]:
    print(f'- {path}')


## 3. V-JEPA 2 Loading and Preprocessing

These cells load the official V-JEPA checkpoint, inspect how Moving MNIST clips are converted into the model's expected RGB / 64-frame format, and confirm that embeddings can be extracted before any ranking or commentary is attempted.

In [ ]:
import json

adapter_config = VJEPA2AdapterConfig(
    model_id=os.environ.get('JEPA_VJEPA_MODEL_ID', 'facebook/vjepa2-vitl-fpc64-256'),
    backend='huggingface',
    fallback_backend='torch_hub',
    fallback_model_name='vjepa2_vit_large',
    cache_dir=str(HF_CACHE_DIR),
)
adapter = VJEPA2Adapter(adapter_config)
vjepa_scorer = VJEPAFutureScorer(adapter=adapter)
print(f'V-JEPA scorer default variant: {vjepa_scorer.scoring_variant}')
print(f'Masked scorer signature: {vjepa_scorer.masked_runtime_signature}')
load_error = None
runtime_info = None

try:
    runtime_info = adapter.load()
    print(json.dumps(runtime_info, indent=2))
except Exception as exc:
    load_error = exc
    print(f'V-JEPA load failed: {exc}')
    if not USE_TORCH_HUB_FALLBACK:
        print('If needed, set JEPA_USE_TORCH_HUB_FALLBACK=1 and rerun the install cell for the official torch.hub fallback path.')

In [ ]:
import json
import torch

if load_error is not None:
    raise RuntimeError(f'Cannot continue preprocessing sanity checks because V-JEPA failed to load: {load_error}')

true_candidate = example.candidates[example.correct_index]
combined_true_clip = torch.cat([example.observed, true_candidate], dim=0)
observed_prep_summary = adapter.summarize_preprocessing(example.observed)
combined_prep_summary = adapter.summarize_preprocessing(combined_true_clip)

print('Observed clip preprocessing summary:')
print(json.dumps(observed_prep_summary, indent=2))
print('\nCombined observed+future clip preprocessing summary:')
print(json.dumps(combined_prep_summary, indent=2))

In [ ]:
import torch

if load_error is not None:
    raise RuntimeError(f'Cannot continue embedding sanity checks because V-JEPA failed to load: {load_error}')

observed_embedding = adapter.encode_video(example.observed)
true_future_embedding = adapter.encode_video(example.candidates[example.correct_index])
combined_embedding = adapter.encode_video(torch.cat([example.observed, example.candidates[example.correct_index]], dim=0))

print('Embedding sanity checks:')
print(f'- observed embedding shape: {tuple(observed_embedding.shape)} | norm={float(torch.linalg.norm(observed_embedding).item()):.4f}')
print(f'- true future embedding shape: {tuple(true_future_embedding.shape)} | norm={float(torch.linalg.norm(true_future_embedding).item()):.4f}')
print(f'- combined clip embedding shape: {tuple(combined_embedding.shape)} | norm={float(torch.linalg.norm(combined_embedding).item()):.4f}')

## 4. Moving MNIST Control Example

The engineered V-JEPA scorer remains the primary representation-based evaluator. This section keeps a single Moving MNIST example in the notebook so we can compare the real-video demo against a known synthetic control.

In [ ]:
import json

if load_error is not None:
    raise RuntimeError(f'Cannot score examples because V-JEPA failed to load: {load_error}')

vjepa_bundle = vjepa_scorer.score_example(
    example.observed,
    example.candidates,
    candidate_metadata=example.metadata['candidate_strategies'],
)
vjepa_candidate_table = build_candidate_score_table(vjepa_bundle, correct_index=example.correct_index)
vjepa_ranking_summary = build_ranking_summary(vjepa_bundle, correct_index=example.correct_index)
vjepa_confidence_tier = getattr(vjepa_bundle, 'confidence_tier', getattr(vjepa_bundle, 'uncertainty', 'unknown'))

single_example_scores = {
    'example_index': example.metadata['index'],
    'correct_index': int(example.correct_index),
    'runtime_info': runtime_info,
    'vjepa_score_bundle': vjepa_bundle.as_dict(),
    'candidate_score_table': vjepa_candidate_table,
    'ranking_summary': vjepa_ranking_summary,
}

single_example_scores_path = vjepa_results_dir / 'single_example_scores.json'
single_example_scores_path.write_text(json.dumps(single_example_scores, indent=2), encoding='utf-8')

print(f'Correct candidate index: {example.correct_index}')
print(f'V-JEPA selected index: {vjepa_bundle.selected_index}')
print(f'Confidence margin: {vjepa_bundle.confidence:.4f} ({vjepa_confidence_tier})')
print('\nCandidate score table:')
for row in vjepa_candidate_table:
    print(json.dumps(row, indent=2))
print(f'\nSaved V-JEPA score bundle to: {single_example_scores_path}')

In [ ]:
import json
from IPython.display import Markdown, display

baseline_result = run_future_selection_pipeline(
    observed=example.observed,
    candidates=example.candidates,
    candidate_metadata=example.metadata['candidate_strategies'],
    evaluation_mode='heuristic',
)

vjepa_pipeline_result = run_future_selection_pipeline(
    observed=example.observed,
    candidates=example.candidates,
    candidate_metadata=example.metadata['candidate_strategies'],
    representation_model=vjepa_scorer,
    evaluation_mode='representation_only',
)

comparison_table = build_baseline_comparison_table(
    vjepa_bundle,
    baseline_result,
    correct_index=example.correct_index,
)
context_payload = build_future_selection_context(
    example,
    vjepa_bundle,
    baseline_bundle=baseline_result,
)

vjepa_commentary = commentary_generator.generate(context_payload)
vjepa_llm_context = llm_ready_builder.build(context_payload)

single_example_trace = {
    'example_index': example.metadata['index'],
    'correct_index': int(example.correct_index),
    'vjepa_pipeline_result': vjepa_pipeline_result.as_dict(),
    'heuristic_pipeline_result': baseline_result.as_dict(),
    'comparison_table': comparison_table,
    'context_payload': context_payload,
    'commentary': vjepa_commentary.as_dict(),
    'llm_ready_commentary_context': vjepa_llm_context.as_dict(),
}

single_example_trace_path = vjepa_results_dir / 'single_example_trace.json'
single_example_trace_path.write_text(json.dumps(single_example_trace, indent=2), encoding='utf-8')
(vjepa_results_dir / 'single_example_commentary.json').write_text(
    json.dumps(vjepa_commentary.as_dict(), indent=2),
    encoding='utf-8',
)
(vjepa_results_dir / 'single_example_commentary.md').write_text(
    vjepa_commentary.to_markdown() + '\n',
    encoding='utf-8',
)
(vjepa_results_dir / 'llm_ready_commentary_context.json').write_text(
    json.dumps(vjepa_llm_context.as_dict(), indent=2),
    encoding='utf-8',
)

print('Heuristic vs engineered V-JEPA comparison on the control benchmark:')
for row in comparison_table:
    print(json.dumps(row, indent=2))
print(f'\nSaved comparison trace to: {single_example_trace_path}')
print('\nGrounded commentary:')
display(Markdown(vjepa_commentary.to_markdown()))

## 5. Real-World Dataset Preparation, V-JEPA Scoring, and LLM Commentary

This section targets Something-Something V2 first through a standard parquet/video ingest path. If that ingest path is unavailable in the current runtime and `JEPA_ALLOW_REAL_VIDEO_FALLBACK` is enabled, the notebook automatically falls back to UCF101. In either case, it caches 16-frame clips locally and runs one real-video example through the engineered V-JEPA scorer and the grounded commentary service.


In [ ]:
import json
from IPython.display import Image as NotebookImage, Markdown, display
from PIL import Image, ImageDraw
import torch

if load_error is not None:
    raise RuntimeError(f'Cannot run the real-video demo because V-JEPA failed to load: {load_error}')

real_adapter = SomethingSomethingV2SubsetAdapter(real_video_config)
real_eval_split = os.environ.get('JEPA_REAL_VIDEO_EVAL_SPLIT', 'test')
force_rebuild_real_cache = os.environ.get('JEPA_REAL_VIDEO_FORCE_REBUILD', '0').strip().lower() in {'1', 'true', 'yes'}
allow_real_video_fallback = os.environ.get('JEPA_ALLOW_REAL_VIDEO_FALLBACK', '1').strip().lower() in {'1', 'true', 'yes'}
real_dataset_label = 'something_something_v2_primary'
real_dataset_note = None

real_video_load_error = None
real_train_manifest = None
real_eval_manifest = None
real_train_dataset = None
real_eval_dataset = None
real_example = None
real_vjepa_bundle = None
real_baseline_result = None
real_candidate_table = []
real_ranking_summary = {}
real_context = {}
real_comparison_table = []
real_commentary_result = None
real_example_panel_path = real_video_results_dir / 'single_example_panel.png'

def _frame_to_rgb(frame):
    frame = torch.as_tensor(frame).detach().cpu().clamp(0.0, 1.0)
    if frame.ndim != 3:
        raise ValueError(f'Expected frame shape (C, H, W); got {tuple(frame.shape)}')
    if frame.shape[0] == 1:
        frame = frame.repeat(3, 1, 1)
    if frame.shape[0] != 3:
        raise ValueError(f'Expected 1 or 3 channels; got {frame.shape[0]}')
    array = frame.mul(255).to(torch.uint8).permute(1, 2, 0).numpy()
    return Image.fromarray(array, mode='RGB')

def _sequence_to_grid(sequence, label, columns=4, padding=2):
    frames = [_frame_to_rgb(frame) for frame in sequence]
    width, height = frames[0].size
    columns = max(1, min(columns, len(frames)))
    rows = (len(frames) + columns - 1) // columns
    grid = Image.new(
        'RGB',
        (columns * width + padding * (columns - 1), rows * height + padding * (rows - 1)),
        color=(0, 0, 0),
    )
    for frame_index, frame in enumerate(frames):
        row = frame_index // columns
        column = frame_index % columns
        x = column * (width + padding)
        y = row * (height + padding)
        grid.paste(frame, (x, y))

    canvas = Image.new('RGB', (grid.width, grid.height + 22), color=(0, 0, 0))
    canvas.paste(grid, (0, 22))
    draw = ImageDraw.Draw(canvas)
    draw.text((4, 3), label, fill=(255, 255, 255))
    return canvas

def _save_real_example_panel(example, path):
    panels = [_sequence_to_grid(example.observed, 'observed (0:8)')]
    for candidate_index, candidate in enumerate(example.candidates):
        meta = example.metadata['candidate_strategies'][candidate_index]
        label = f"candidate {candidate_index} | {meta.get('strategy', 'unknown')}"
        panels.append(_sequence_to_grid(candidate, label))

    width = max(panel.width for panel in panels)
    height = sum(panel.height for panel in panels) + 4 * (len(panels) - 1)
    canvas = Image.new('RGB', (width, height), color=(0, 0, 0))
    y = 0
    for panel in panels:
        canvas.paste(panel, (0, y))
        y += panel.height + 4
    canvas.save(path)
    return path

try:
    real_train_manifest = real_adapter.prepare_cache('train', force_rebuild=force_rebuild_real_cache)
    real_eval_manifest = real_adapter.prepare_cache(real_eval_split, force_rebuild=force_rebuild_real_cache)
    real_train_dataset = real_adapter.build_dataset('train')
    real_eval_dataset = real_adapter.build_dataset(real_eval_split)
    real_example = real_eval_dataset[0]
    _save_real_example_panel(real_example, real_example_panel_path)
except Exception as exc:
    if allow_real_video_fallback and not real_video_config.local_manifest_path:
        fallback_config = make_ucf101_fallback_config(real_video_config)
        real_adapter = UCF101SubsetAdapter(fallback_config)
        real_dataset_label = 'ucf101_fallback'
        real_dataset_note = f'Primary Something-Something ingest failed: {exc}'
        try:
            real_train_manifest = real_adapter.prepare_cache('train', force_rebuild=force_rebuild_real_cache)
            real_eval_manifest = real_adapter.prepare_cache(real_eval_split, force_rebuild=force_rebuild_real_cache)
            real_train_dataset = real_adapter.build_dataset('train')
            real_eval_dataset = real_adapter.build_dataset(real_eval_split)
            real_example = real_eval_dataset[0]
            _save_real_example_panel(real_example, real_example_panel_path)
        except Exception as fallback_exc:
            real_video_load_error = fallback_exc
    else:
        real_video_load_error = exc

if real_video_load_error is not None:
    raise RuntimeError(
        'Real-video dataset preparation failed. Provide a local manifest with JEPA_REAL_VIDEO_MANIFEST if needed. '
        'The notebook now targets the parquet/video Something-Something V2 path first and falls back to UCF101 when fallback is enabled.'
    ) from real_video_load_error

print('Real-video cache manifests:')
print(f'- dataset backend: {real_dataset_label}')
print(f'- dataset source: {real_adapter.config.dataset_name}')
if real_dataset_note is not None:
    print(f'- note: {real_dataset_note}')
print(f'- train: {real_train_manifest}')
print(f'- {real_eval_split}: {real_eval_manifest}')
print(f'- train examples: {len(real_train_dataset)}')
print(f'- {real_eval_split} examples: {len(real_eval_dataset)}')
print(f'- commentary backend: {getattr(commentary_service.backend, "name", type(commentary_service.backend).__name__)}')
display(NotebookImage(filename=str(real_example_panel_path)))

real_vjepa_bundle = vjepa_scorer.score_example(
    real_example.observed,
    real_example.candidates,
    candidate_metadata=real_example.metadata['candidate_strategies'],
)
real_baseline_result = run_future_selection_pipeline(
    observed=real_example.observed,
    candidates=real_example.candidates,
    candidate_metadata=real_example.metadata['candidate_strategies'],
    evaluation_mode='heuristic',
)
real_candidate_table = build_candidate_score_table(real_vjepa_bundle, correct_index=real_example.correct_index)
real_ranking_summary = build_ranking_summary(real_vjepa_bundle, correct_index=real_example.correct_index)
real_context = build_future_selection_context(real_example, real_vjepa_bundle, baseline_bundle=real_baseline_result)
real_comparison_table = build_baseline_comparison_table(
    real_vjepa_bundle,
    real_baseline_result,
    correct_index=real_example.correct_index,
)
real_commentary_result = generate_commentary(
    real_context,
    service=commentary_service,
    expected_selected_index=int(real_vjepa_bundle.selected_index),
)

real_single_example_scores = {
    'source_video_id': real_example.metadata['source_video_id'],
    'correct_index': int(real_example.correct_index),
    'runtime_info': runtime_info,
    'dataset_backend': real_dataset_label,
    'dataset_source': real_adapter.config.dataset_name,
    'dataset_note': real_dataset_note,
    'vjepa_score_bundle': real_vjepa_bundle.as_dict(),
    'candidate_score_table': real_candidate_table,
    'ranking_summary': real_ranking_summary,
    'context_payload': real_context,
}
real_single_example_trace = {
    'source_video_id': real_example.metadata['source_video_id'],
    'correct_index': int(real_example.correct_index),
    'dataset_backend': real_dataset_label,
    'dataset_source': real_adapter.config.dataset_name,
    'dataset_note': real_dataset_note,
    'comparison_table': real_comparison_table,
    'commentary_result': real_commentary_result.as_dict(),
}

(real_video_results_dir / 'single_example_scores.json').write_text(
    json.dumps(real_single_example_scores, indent=2),
    encoding='utf-8',
)
(real_video_results_dir / 'single_example_trace.json').write_text(
    json.dumps(real_single_example_trace, indent=2),
    encoding='utf-8',
)
(real_video_results_dir / 'single_example_commentary.json').write_text(
    json.dumps(real_commentary_result.as_dict(), indent=2),
    encoding='utf-8',
)
(real_video_results_dir / 'single_example_commentary.md').write_text(
    real_commentary_result.commentary.to_markdown() + '\n',
    encoding='utf-8',
)
(real_video_results_dir / 'llm_ready_commentary_context.json').write_text(
    json.dumps(real_commentary_result.llm_context, indent=2),
    encoding='utf-8',
)

print('Real-video heuristic vs engineered V-JEPA comparison:')
for row in real_comparison_table:
    print(json.dumps(row, indent=2))
print('\nReal-video grounded commentary:')
display(Markdown(real_commentary_result.commentary.to_markdown()))


## 6. Live Agent Mode and Real-World Evaluation

The live agent loop processes examples one at a time, prints the selected future plus grounded commentary, and saves transcript artifacts. After that, the notebook runs a larger real-world evaluation slice so the real-video path is measured on more than a single example.

In [ ]:
import json

if real_video_load_error is not None or real_eval_dataset is None:
    raise RuntimeError('Real-video evaluation is unavailable because dataset preparation failed earlier.')

real_live_count = min(int(os.environ.get('JEPA_LIVE_AGENT_COUNT', '5')), len(real_eval_dataset))
real_eval_count = min(int(os.environ.get('JEPA_REAL_VIDEO_EVAL_COUNT', '128')), len(real_eval_dataset))
print(
    f'Real-video demo defaults: live_count={real_live_count}, evaluation_count={real_eval_count}. '
    'Set JEPA_LIVE_AGENT_COUNT or JEPA_REAL_VIDEO_EVAL_COUNT for a different slice.'
)

real_live_result = run_live_agent_loop(
    real_eval_dataset,
    vjepa_scorer,
    count=real_live_count,
    output_dir=agent_live_results_dir,
    commentary_service=commentary_service,
    include_heuristic_baseline=True,
    progress=True,
)
(agent_live_results_dir / 'latest_run_summary.json').write_text(
    json.dumps(real_live_result.as_dict(), indent=2),
    encoding='utf-8',
)

real_benchmark_result = run_future_selection_benchmark(
    real_eval_dataset,
    evaluators={
        'heuristic': None,
        'vjepa': vjepa_scorer,
    },
    evaluation_count=real_eval_count,
    show_progress=True,
    progress_interval=max(1, real_eval_count // 8),
)
real_benchmark_artifacts = save_future_selection_benchmark_artifacts(
    real_benchmark_result,
    real_video_results_dir,
    report_path=report_dir / 'real_video_eval_summary.md',
    title='Real-World Future-Selection Evaluation Summary',
    metadata={
        'dataset_name': real_video_config.dataset_name,
        'eval_split': real_eval_split,
        'cache_root': str((REPO_ROOT / real_video_config.cache_root).resolve()),
        'backend_used': runtime_info['backend_used'] if runtime_info else 'unknown',
        'commentary_backend': getattr(commentary_service.backend, 'name', type(commentary_service.backend).__name__),
        'live_agent_count': real_live_count,
    },
)

print('Real-world benchmark summary:')
print(json.dumps(real_benchmark_result.summary, indent=2))
print('\nPer-negative-type summary:')
print(json.dumps(real_benchmark_result.per_negative_type, indent=2))
print('\nLive agent artifacts:')
print(f'- jsonl: {real_live_result.jsonl_path}')
print(f'- markdown: {real_live_result.markdown_path}')
print('\nSaved real-world benchmark artifacts:')
for name, path in real_benchmark_artifacts.items():
    print(f'- {name}: {path}')

## 7. Saved Artifacts and Export

This final section surfaces both the Moving MNIST control artifacts and the new real-video artifacts, then packages everything into one archive so results can be pulled back from the remote runtime reliably.

In [ ]:
import os
os.environ['JEPA_DRIVE_EXPORT_DIR'] = '/content/drive/MyDrive/jepa_exports'
# Optional for PyCharm-connected Colab kernels: disable browser download and rely on Drive copy.
os.environ.setdefault('JEPA_DOWNLOAD_ARTIFACT_BUNDLE', '0')

In [ ]:
from pathlib import Path
from datetime import datetime
from IPython.display import Image as NotebookImage, display
import json
import os
import shutil
import zipfile

artifact_export_dir = REPO_ROOT / 'results' / 'artifact_exports'
artifact_export_dir.mkdir(parents=True, exist_ok=True)

artifact_roots = [
    REPO_ROOT / 'results' / 'moving_mnist',
    REPO_ROOT / 'results' / 'task_examples',
    REPO_ROOT / 'results' / 'vjepa_eval',
    REPO_ROOT / 'results' / 'real_video_eval',
    REPO_ROOT / 'results' / 'agent_live',
    REPO_ROOT / 'report',
]

for root in artifact_roots:
    print(f'Artifacts under {root}:')
    if not root.exists():
        print('  - missing')
        continue
    for path in sorted(root.iterdir()):
        print(f'  - {path.name}')

panel_paths = [
    *(task_results_dir.glob('*_panel.png')),
    *(real_video_results_dir.glob('*panel*.png')),
]
if panel_paths:
    display(NotebookImage(filename=str(sorted(panel_paths)[0])))

commentary_paths = [
    vjepa_results_dir / 'single_example_commentary.md',
    real_video_results_dir / 'single_example_commentary.md',
    report_dir / 'vjepa_eval_summary.md',
    report_dir / 'real_video_eval_summary.md',
]
latest_live_markdown = sorted(agent_live_results_dir.glob('*.md'))
if latest_live_markdown:
    commentary_paths.append(latest_live_markdown[-1])

for path in commentary_paths:
    if path.exists():
        print(f'\nRendered text from {path.name}:\n')
        print(path.read_text(encoding='utf-8'))

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
bundle_path = artifact_export_dir / f'jepa_demo_artifacts_{timestamp}.zip'
included_files = []
total_bytes = 0

with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for root in artifact_roots:
        if not root.exists():
            continue
        for file_path in sorted(p for p in root.rglob('*') if p.is_file()):
            arcname = file_path.relative_to(REPO_ROOT).as_posix()
            bundle.write(file_path, arcname)
            included_files.append(arcname)
            total_bytes += int(file_path.stat().st_size)

bundle_summary = {
    'archive_path': str(bundle_path),
    'file_count': len(included_files),
    'total_bytes': total_bytes,
    'included_files': included_files,
}
print(f'\nCreated artifact bundle: {bundle_path}')
print(json.dumps(bundle_summary, indent=2))

drive_export_dir = os.environ.get('JEPA_DRIVE_EXPORT_DIR', '').strip()
if drive_export_dir:
    drive_target = Path(drive_export_dir).expanduser()
    if str(drive_target).startswith('/content/drive'):
        try:
            from google.colab import drive
            if not Path('/content/drive/MyDrive').exists():
                print('Mounting Google Drive for artifact export...')
                drive.mount('/content/drive', force_remount=False)
        except Exception as exc:
            print(f'Unable to mount Google Drive automatically: {exc}')
    drive_target.mkdir(parents=True, exist_ok=True)
    copied_path = drive_target / bundle_path.name
    shutil.copy2(bundle_path, copied_path)
    print(f'Copied artifact bundle to: {copied_path}')

try:
    from google.colab import files
    if os.environ.get('JEPA_DOWNLOAD_ARTIFACT_BUNDLE', '1').strip().lower() in {'1', 'true', 'yes'}:
        print('Attempting browser download of the artifact bundle...')
        files.download(str(bundle_path))
except Exception as exc:
    print(f'Automatic download unavailable in this notebook client: {exc}')

if not drive_export_dir:
    print('No JEPA_DRIVE_EXPORT_DIR is set. In PyCharm-connected Colab sessions, browser download may not bring the file back to your local machine.')
    print('Set JEPA_DRIVE_EXPORT_DIR to a mounted Drive folder if you want a reliable export path.')